In [5]:
import pandas as pd
import numpy as np

# 1. Load raw dataset
df = pd.read_csv('train.csv')
print("Initial shape:", df.shape)

# 2. Drop exact duplicate rows
df = df.drop_duplicates()

# 3. Standardize column names (lowercase, spaces to underscores)
df.columns = [col.strip().lower().replace(' ', '_') for col in df.columns]

# 4. Audit missing values before cleaning
print("Missing values before cleaning:\n", df.isnull().sum())

# 5. SURGICAL DROP: Drop rows missing critical categorical markers (star_type or water_presence)
df = df.dropna(subset=['star_type', 'water_presence'])
print(f"Row count after dropping missing critical categorical records: {len(df)}")

# 6. HANDLE MOONS & RINGS: Unrecorded usually means none, fill with 0
df['moons_count'] = df['moons_count'].fillna(0)
df['rings'] = df['rings'].fillna(0)

# 7. MULTI-FACTOR SMART IMPUTATION: Fill missing numerical telemetry
# using the median grouped by BOTH star_type AND planet_type for maximum physical accuracy
numerical_cols = df.select_dtypes(include=['float64', 'int64']).columns
for col in numerical_cols:
    if df[col].isnull().sum() > 0:
        # Group by both star_type and planet_type
        df[col] = df[col].fillna(
            df.groupby(['star_type', 'planet_type'])[col].transform('median')
        )
        # Fallback to planet_type median if star+planet group is NaN
        df[col] = df[col].fillna(
            df.groupby('planet_type')[col].transform('median')
        )
        # Final fallback to overall median
        df[col] = df[col].fillna(df[col].median())

# 8. Final check on missing values (should be completely 0 across all columns!)
print("\nMissing values after complete cleaning:\n", df.isnull().sum())

# 9. Export the cleaned dataset
df.to_csv('InfoReaper_cleaned_dataset.csv', index=False)
print(f"Cleaned dataset successfully saved as InfoReaper_cleaned_dataset.csv! Final row count: {len(df)}")

Initial shape: (6400, 31)
Missing values before cleaning:
 planetid                       0
planet_name                    0
star_type                      1
planet_type                    0
orbital_distance_au          637
orbital_period_days          639
day_length_hours              56
axial_tilt_deg                48
mean_temp_c                    0
temp_range_c                   0
gravity_g                      0
escape_velocity_kms          642
atmospheric_pressure_atm       0
atmosphere_primary_gases       0
oxygen_percent               972
co2_percent                 1210
methane_percent              882
water_presence                 4
surface_composition            0
albedo                         0
cloud_coverage_percent         0
storm_frequency                0
seismic_activity               0
magnetic_field                 0
radiation_level                0
moons_count                    7
rings                          7
rare_mineral_index             0
energy_harvest_po